# Unit 8 (Part 1) — PPO from scratch (CleanRL style) on LunarLander-v2

**Pass criterion:** `mean_reward − std_reward ≥ −500` (deliberately low — the goal is to **implement** PPO yourself)

In Unit 1 we used PPO from Stable-Baselines3. Here we write **the same algorithm line by line** (~150 lines), with each block explained.

**How to run:** `Runtime → Run all` (~20–30 min, no GPU). The Hugging Face login reads the `HF_TOKEN` Colab secret.

In [ ]:
!apt-get install -y -qq swig cmake ffmpeg > /dev/null
!pip install -q "gymnasium[box2d]==0.29.1" huggingface_hub imageio imageio-ffmpeg tensorboard

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ENV_ID = "LunarLander-v2"
REPO_ID = f"{HF_USERNAME}/ppo-scratch-{ENV_ID}"

import datetime, json, random, tempfile, time
from pathlib import Path
from types import SimpleNamespace
import imageio
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter
import gymnasium as gym

import base64
from IPython.display import HTML, display

def show_video(path, width=500):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## 1. Hyperparameters
| Name | Meaning |
|---|---|
| `num_envs`, `num_steps` | `num_envs × num_steps` transitions are collected before each update (**rollout**) |
| `gamma` (γ) | Discount factor |
| `gae_lambda` (λ) | GAE: how far ahead the advantage estimate looks |
| `num_minibatches`, `update_epochs` | How many minibatches the rollout is split into and how many passes are made over it |
| `clip_coef` (ε) | **Clipping:** the probability ratio is kept inside `[1−ε, 1+ε]` |
| `ent_coef` | Entropy bonus → exploration |
| `vf_coef` | Weight of the critic (value) loss |
| `max_grad_norm` | Gradient clipping → prevents sudden blow-ups |

In [ ]:
args = SimpleNamespace(
    env_id=ENV_ID,
    seed=1,
    total_timesteps=1_000_000,
    learning_rate=3e-4,
    num_envs=16,
    num_steps=256,
    anneal_lr=True,
    gamma=0.999,
    gae_lambda=0.98,
    num_minibatches=32,
    update_epochs=4,
    norm_adv=True,
    clip_coef=0.2,
    clip_vloss=True,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
)
args.batch_size = args.num_envs * args.num_steps           # 4096 steps per update
args.minibatch_size = args.batch_size // args.num_minibatches
args.num_updates = args.total_timesteps // args.batch_size
print("batch:", args.batch_size, "| minibatch:", args.minibatch_size, "| updates:", args.num_updates)

random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
device = torch.device("cpu")

## 2. Vectorised environments
`RecordEpisodeStatistics` records the total return of every episode (for logging).

In [ ]:
def make_env(env_id, seed):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env.action_space.seed(seed)
        return env
    return thunk

envs = gym.vector.SyncVectorEnv([make_env(args.env_id, args.seed + i) for i in range(args.num_envs)])
obs_dim = envs.single_observation_space.shape[0]
n_actions = envs.single_action_space.n
print("obs:", obs_dim, "actions:", n_actions)

## 3. Agent = Actor + Critic
- **Critic** `V(s)`: "how good is this state?" → a single number.
- **Actor** `π(a|s)`: probabilities of the 4 actions (logits → Categorical).
- Weights use **orthogonal** initialisation (standard for PPO; stable start).

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self):
        super().__init__()
        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )
        self.actor = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, n_actions), std=0.01),
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        logits = self.actor(x)
        probs = Categorical(logits=logits)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action), probs.entropy(), self.critic(x)

agent = Agent().to(device)
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)

## 4. Live training dashboard: TensorBoard
Refresh with ⟳ during training. Watch `charts/episodic_return` (should rise), `losses/approx_kl` and `losses/clipfrac` (how often clipping kicks in).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. PPO training loop
Four stages per update:
1. **Rollout:** play `num_steps` steps with the current policy; store states, actions, log-probs, rewards and values.
2. **Advantages with GAE:** `δ_t = r_t + γ·V(s_{t+1}) − V(s_t)`, `A_t = δ_t + γλ·A_{t+1}` (computed backwards).
3. **Clipped surrogate loss:** `ratio = π_new / π_old = exp(log π_new − log π_old)`
   `L_actor = max(−A·ratio, −A·clip(ratio, 1−ε, 1+ε))`
4. **Total loss:** `L = L_actor − ent_coef·Entropy + vf_coef·L_critic` → `backward()`

In [ ]:
run_name = f"{args.env_id}__ppo_scratch__{args.seed}__{int(time.time())}"
writer = SummaryWriter(f"runs/{run_name}")

# Rollout storage
obs      = torch.zeros((args.num_steps, args.num_envs, obs_dim)).to(device)
actions  = torch.zeros((args.num_steps, args.num_envs)).to(device)
logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
rewards  = torch.zeros((args.num_steps, args.num_envs)).to(device)
dones    = torch.zeros((args.num_steps, args.num_envs)).to(device)
values   = torch.zeros((args.num_steps, args.num_envs)).to(device)

global_step = 0
start_time = time.time()
next_obs, _ = envs.reset(seed=args.seed)
next_obs = torch.Tensor(next_obs).to(device)
next_done = torch.zeros(args.num_envs).to(device)
recent_returns = []

for update in range(1, args.num_updates + 1):
    # Learning-rate annealing: linearly down to 0
    if args.anneal_lr:
        frac = 1.0 - (update - 1.0) / args.num_updates
        optimizer.param_groups[0]["lr"] = frac * args.learning_rate

    # ---- 1) ROLLOUT -------------------------------------------------------
    for step in range(args.num_steps):
        global_step += args.num_envs
        obs[step] = next_obs
        dones[step] = next_done
        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(next_obs)
            values[step] = value.flatten()
        actions[step] = action
        logprobs[step] = logprob

        next_obs, reward, terminated, truncated, infos = envs.step(action.cpu().numpy())
        done = np.logical_or(terminated, truncated)
        rewards[step] = torch.tensor(reward).to(device).view(-1)
        next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(done).to(device)

        if "final_info" in infos:
            for info in infos["final_info"]:
                if info and "episode" in info:
                    r = float(np.ravel(info["episode"]["r"])[0])
                    recent_returns.append(r)
                    writer.add_scalar("charts/episodic_return", r, global_step)

    # ---- 2) GAE -----------------------------------------------------------
    with torch.no_grad():
        next_value = agent.get_value(next_obs).reshape(1, -1)
        advantages = torch.zeros_like(rewards).to(device)
        lastgaelam = 0
        for t in reversed(range(args.num_steps)):
            if t == args.num_steps - 1:
                nextnonterminal = 1.0 - next_done
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - dones[t + 1]
                nextvalues = values[t + 1]
            delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
            advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
        returns = advantages + values

    # Flatten the batch
    b_obs = obs.reshape((-1, obs_dim))
    b_logprobs = logprobs.reshape(-1)
    b_actions = actions.reshape(-1)
    b_advantages = advantages.reshape(-1)
    b_returns = returns.reshape(-1)
    b_values = values.reshape(-1)

    # ---- 3) + 4) OPTIMIZE -------------------------------------------------
    b_inds = np.arange(args.batch_size)
    clipfracs = []
    for epoch in range(args.update_epochs):
        np.random.shuffle(b_inds)
        for start in range(0, args.batch_size, args.minibatch_size):
            mb = b_inds[start:start + args.minibatch_size]

            _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb], b_actions.long()[mb])
            logratio = newlogprob - b_logprobs[mb]
            ratio = logratio.exp()                                   # pi_new / pi_old

            with torch.no_grad():
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs.append(((ratio - 1.0).abs() > args.clip_coef).float().mean().item())

            mb_adv = b_advantages[mb]
            if args.norm_adv:
                mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

            # Actor: clipped surrogate objective (the "seat belt")
            pg_loss1 = -mb_adv * ratio
            pg_loss2 = -mb_adv * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()

            # Critic: (clipped) value loss
            newvalue = newvalue.view(-1)
            if args.clip_vloss:
                v_unclipped = (newvalue - b_returns[mb]) ** 2
                v_clipped = b_values[mb] + torch.clamp(newvalue - b_values[mb], -args.clip_coef, args.clip_coef)
                v_loss = 0.5 * torch.max(v_unclipped, (v_clipped - b_returns[mb]) ** 2).mean()
            else:
                v_loss = 0.5 * ((newvalue - b_returns[mb]) ** 2).mean()

            entropy_loss = entropy.mean()
            loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
            optimizer.step()

    writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
    writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
    writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
    writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
    writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
    sps = int(global_step / (time.time() - start_time))
    writer.add_scalar("charts/SPS", sps, global_step)
    if update % 10 == 0 or update == args.num_updates:
        avg = np.mean(recent_returns[-50:]) if recent_returns else float("nan")
        print(f"update {update:4d}/{args.num_updates} | step {global_step:8d} | "
              f"avg return (last 50 eps) {avg:8.2f} | clipfrac {np.mean(clipfracs):.3f} | {sps} steps/s")

envs.close()
writer.close()

## 6. Evaluate (10 episodes)

In [ ]:
def evaluate(agent, n_episodes=10, seed=100):
    env = gym.make(ENV_ID)
    rets = []
    for i in range(n_episodes):
        o, _ = env.reset(seed=seed + i)
        total, done = 0.0, False
        while not done:
            with torch.no_grad():
                logits = agent.actor(torch.Tensor(o).unsqueeze(0))
            o, r, term, trunc, _ = env.step(int(logits.argmax()))
            total += r
            done = term or trunc
        rets.append(total)
    env.close()
    return float(np.mean(rets)), float(np.std(rets))

mean_reward, std_reward = evaluate(agent)
score = mean_reward - std_reward
print(f"mean_reward = {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"score (mean - std) = {score:.2f}  ->  {'PASSED ✅' if score >= -500 else 'NOT YET ❌'}")

## 7. Watch your own PPO agent

In [ ]:
def record(agent, path="replay.mp4", seed=0):
    env = gym.make(ENV_ID, render_mode="rgb_array")
    o, _ = env.reset(seed=seed)
    frames, total, done = [env.render()], 0.0, False
    while not done:
        with torch.no_grad():
            a = int(agent.actor(torch.Tensor(o).unsqueeze(0)).argmax())
        o, r, term, trunc, _ = env.step(a)
        total += r
        frames.append(env.render())
        done = term or trunc
    env.close()
    imageio.mimsave(path, frames, fps=30, macro_block_size=1)
    print(f"Episode reward: {total:.1f}")
    return path

show_video(record(agent))

## 8. Push to the Hugging Face Hub

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.repocard import metadata_eval_result, metadata_save
import shutil

def package_to_hub(repo_id, agent, args, logs_dir):
    api = HfApi()
    api.create_repo(repo_id=repo_id, exist_ok=True)
    repo_name = repo_id.split("/")[1]
    mean_reward, std_reward = evaluate(agent)
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        torch.save(agent.state_dict(), tmp / "model.pt")
        json.dump({"env_id": args.env_id, "mean_reward": mean_reward, "std_reward": std_reward,
                   "n_evaluation_episodes": 10, "eval_datetime": datetime.datetime.now().isoformat()},
                  open(tmp / "results.json", "w"))
        record(agent, str(tmp / "replay.mp4"))
        metadata = {"tags": [args.env_id, "ppo", "deep-reinforcement-learning", "reinforcement-learning",
                             "custom-implementation", "deep-rl-course"]}
        metadata.update(metadata_eval_result(
            model_pretty_name=repo_name, task_pretty_name="reinforcement-learning",
            task_id="reinforcement-learning", metrics_pretty_name="mean_reward",
            metrics_id="mean_reward", metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
            dataset_pretty_name=args.env_id, dataset_id=args.env_id))
        hp = "\n".join(f"{k}: {v}" for k, v in vars(args).items())
        readme = tmp / "README.md"
        readme.write_text(f"# PPO Agent Playing {args.env_id}\n\n"
                          f"PPO implemented from scratch (CleanRL style) for Unit 8 of the Deep RL Course.\n\n"
                          f"## Hyperparameters\n```\n{hp}\n```\n", encoding="utf-8")
        metadata_save(readme, metadata)
        shutil.copytree(logs_dir, tmp / "logs")
        api.upload_folder(repo_id=repo_id, folder_path=tmp, path_in_repo=".")
    print(f"Done: https://huggingface.co/{repo_id}  ({mean_reward:.2f} +/- {std_reward:.2f})")

package_to_hub(REPO_ID, agent, args, f"runs/{run_name}")

## 9. Check

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")

check_like_certificate("LunarLander-v2", "deep-rl-course", -500)